In [1]:
# Импортируем алхимию и создаем соединение
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np

load_dotenv()

POSTGRES_USER = os.getenv('POSTGRES_USER')
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD')
POSTGRES_DB = os.getenv('POSTGRES_DB')
POSTGRES_HOST= os.getenv('POSTGRES_HOST', 'localhost')

conn = create_engine(f'postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:5432/{POSTGRES_DB}')


In [2]:
sql = '''
    SELECT 
        *
    FROM raw.extract_api_jobs
'''

df = pd.read_sql(sql, conn)
df

,id,endpoint,request_payload,response_data,status_code,loaded_at
0,1,MarketDataService/GetCandles,"{'to': '2026-06-30T06:08:41.455247+00:00', 'fi...","{'candles': [{'low': {'nano': 230000000, 'unit...",200,2026-06-30 06:08:41.585007+00:00
1,2,MarketDataService/GetCandles,"{'to': '2026-06-30T06:39:42.378524+00:00', 'fi...","{'candles': [{'low': {'nano': 0, 'units': '297...",200,2026-06-30 06:39:42.553008+00:00
2,3,MarketDataService/GetCandles,"{'to': '2026-07-01T05:15:45.774673+00:00', 'fi...","{'candles': [{'low': {'nano': 890000000, 'unit...",200,2026-07-01 05:15:45.932152+00:00
3,4,MarketDataService/GetCandles,"{'to': '2026-07-02T05:15:47.038426+00:00', 'fi...","{'candles': [{'low': {'nano': 850000000, 'unit...",200,2026-07-02 05:15:51.891103+00:00
4,5,MarketDataService/GetCandles,"{'to': '2026-07-02T05:15:49.436653+00:00', 'fi...","{'candles': [{'low': {'nano': 850000000, 'unit...",200,2026-07-02 05:15:54.170052+00:00


In [3]:
sql = '''
select
    id,
    jsonb_array_length(response_data -> 'candles') AS resp_candles_cnt,
    request_payload #>> '{from}' as req_from_dttm,
	request_payload #>> '{to}' as req_to_dttm,
	request_payload #>> '{figi}' as req_figi,
	request_payload #>> '{interval}' as req_interval,
	status_code,
	loaded_at as db_loaded_dttm
from raw.extract_api_jobs
where endpoint = 'MarketDataService/GetCandles'
and status_code = 200;
'''


df = pd.read_sql(sql, conn)
df

,id,resp_candles_cnt,req_from_dttm,req_to_dttm,req_figi,req_interval,status_code,db_loaded_dttm
0,1,70,2026-06-29T06:08:41.455247+00:00,2026-06-30T06:08:41.455247+00:00,BBG004730N88,CANDLE_INTERVAL_15_MIN,200,2026-06-30 06:08:41.585007+00:00
1,2,70,2026-06-29T06:39:42.378524+00:00,2026-06-30T06:39:42.378524+00:00,BBG004730N88,CANDLE_INTERVAL_15_MIN,200,2026-06-30 06:39:42.553008+00:00
2,3,69,2026-06-30T05:15:45.774673+00:00,2026-07-01T05:15:45.774673+00:00,BBG004730N88,CANDLE_INTERVAL_15_MIN,200,2026-07-01 05:15:45.932152+00:00
3,4,69,2026-07-01T05:15:47.038426+00:00,2026-07-02T05:15:47.038426+00:00,BBG004730N88,CANDLE_INTERVAL_15_MIN,200,2026-07-02 05:15:51.891103+00:00
4,5,69,2026-07-01T05:15:49.436653+00:00,2026-07-02T05:15:49.436653+00:00,BBG004730N88,CANDLE_INTERVAL_15_MIN,200,2026-07-02 05:15:54.170052+00:00
